# 02 - Exploratory data analysis: univariate and bivariate (household level)

DRAFT code written by Claude (AI assistant) for study and verification. Fill in the Purpose / What the code does / My result fields in your own words after you have run and understood each cell.

Input: `household_table.csv` from notebook 01 (one-family households, major income earner's row). Output: tables (CSV) and figures (PNG) saved to an `EDA_outputs` folder, to be copied into the GitHub repo.

Counts and percentages are unweighted (the households in the sample) unless a column or chart says weighted (weighted by FWEIGHT, to compare with published prevalence). All nine predictors are treated as categories (income quintile, tenure, has_children and edu_household were built in notebook 01; HHSIZE is also treated as a category here), so no correlation of codes as if they were quantities is computed.

### Step 1 - Libraries, folders, and loading the table

**Purpose:** 

**What the code does:** 

**Learn from:** [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [nullable integers](https://pandas.pydata.org/docs/user_guide/integer_na.html)

**My result:** 

In [ ]:
# ================= SETTINGS: the only lines to change when reproducing this work =================
# 1. Download the five Canadian Income Survey public-use files (CIS2018_PUMF.txt ... CIS2022_PUMF.txt) from Statistics Canada
#    and put them in a folder called ProjectData inside base_path. They are large files, so they are not stored on GitHub.
# 2. Google Colab with Google Drive: keep the path below, or change it to wherever you put the folder.
#    Your own computer (no Drive): replace it with a local folder, e.g. './CIND820_CapstoneProject'.
# 3. Run notebook 01, then 02. The results are written to ProcessedData and EDA_outputs inside base_path.
base_path = '/content/drive/MyDrive/CIND820_CapstoneProject'
# ===================================================================================================

import os
import textwrap                          # wraps long labels so chart axes stay readable (standard library)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # charts
from scipy.stats import chi2_contingency                # chi-square test of independence
from scipy.stats.contingency import association         # Cramer's V (effect size for two categorical variables)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

input_file = f'{base_path}/ProcessedData/household_table.csv'      # made by notebook 01
missing_file = f'{base_path}/ProcessedData/missingness_by_year.csv'
outputs_dir = f'{base_path}/EDA_outputs'                           # copy this folder into the GitHub repo afterwards
os.makedirs(outputs_dir, exist_ok=True)

# Must match the predictors list in notebook 01 (remove or add names to change the analysis)
predictors = ['income_quintile', 'tenure', 'has_children', 'SEX', 'edu_household', 'IMMST', 'MAJRI', 'PROV', 'HHSIZE']
years = [2018, 2019, 2020, 2021, 2022]

# Everything except the weight is a code, so read it as a nullable integer (Int64: whole numbers that may be missing).
# This is the type check the Milestone 2 page asks for: profiling tools can treat IDs and codes as ordinary numbers.
code_cols = ['YEAR', 'PUMFID', 'FSCHHLDM', 'food_insecure'] + predictors
hh = pd.read_csv(input_file, dtype={c: 'Int64' for c in code_cols}, usecols=lambda c: c in code_cols + ['FWEIGHT'])
# FWEIGHT (the major income earner's person weight) times the 0/1 target: used below for weighted food insecurity rates
hh['weight_x_insecure'] = hh['FWEIGHT'] * hh['food_insecure']
print(hh.shape)
print(hh.dtypes)

### Step 2 - Labels from the data dictionaries (so tables show words, not only codes)

**Purpose:** 

**What the code does:** 

**Learn from:** The 'Answer Categories' of each variable in the data dictionaries

**My result:** 

In [ ]:
# Code -> label. PROV, HLEV2G, IMMST, MAJRI and FSCHHLDM are copied from the answer categories in the dictionaries
# (identical in all five years); SEX likewise. income_quintile, tenure, has_children and edu_household were defined in notebook 01.
labels = {'PROV': {10: 'Newfoundland and Labrador',
          11: 'Prince Edward Island',
          12: 'Nova Scotia',
          13: 'New Brunswick',
          24: 'Quebec',
          35: 'Ontario',
          46: 'Manitoba',
          47: 'Saskatchewan',
          48: 'Alberta',
          59: 'British Columbia'},
 'HLEV2G': {1: 'Less than high school graduation',
            2: 'Graduated high school or partial postsecondary education',
            3: 'Non-university postsecondary certificate or diploma',
            4: 'University degree or certificate',
            6: 'Valid skip',
            9: 'Not stated'},
 'IMMST': {1: 'Yes', 2: 'No', 6: 'Valid skip', 9: 'Not stated'},
 'FSCHHLDM': {0: 'Food secure',
              1: 'Marginally food insecure',
              2: 'Moderately food insecure',
              3: 'Severely food insecure'},
 'MAJRI': {1: 'No income',
           2: 'Wages and salaries',
           3: 'Self-employment income',
           4: 'Government transfers',
           5: 'Investment income',
           6: 'Private retirement pensions',
           7: 'Other income',
           96: 'Valid skip'},
 'income_quintile': {1: 'Lowest fifth',
                     2: 'Second fifth',
                     3: 'Middle fifth',
                     4: 'Fourth fifth',
                     5: 'Highest fifth'},
 'tenure': {1: 'Not owned', 2: 'Owned, with mortgage', 3: 'Owned, no mortgage'},
 'has_children': {0: 'Adults only', 1: 'At least one person under 18'},
 'edu_household': {1: 'Less than high school graduation',
                   2: 'Graduated high school or partial postsecondary education',
                   3: 'Non-university postsecondary certificate or diploma',
                   4: 'University degree or certificate'},
 'SEX': {1: 'Male', 2: 'Female'}}


def label(var, code):
    """'35 - Ontario' for coded variables; the plain number for HHSIZE."""
    if var in labels and int(code) in labels[var]:
        return f'{int(code)} - {labels[var][int(code)]}'
    return str(int(code))

def wrap(text, width=38):
    return '\n'.join(textwrap.wrap(str(text), width))

year_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(years)))   # ordered colours: darker = earlier year

def save_fig(fig, name):
    fig.savefig(f'{outputs_dir}/{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 1. Target: food security status of the household

### Step 3 - Target by year (counts, percentages, binary target)

**Purpose:** 

**What the code does:** 

**Learn from:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [numpy.average](https://numpy.org/doc/stable/reference/generated/numpy.average.html), [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

**My result:** 

In [ ]:
# --- Households in each FSCHHLDM category, by year ---
counts = pd.crosstab(hh['FSCHHLDM'], hh['YEAR'])          # rows = category, columns = year
counts.index = [label('FSCHHLDM', c) for c in counts.index]
counts.loc['Total households'] = counts.sum()
display(counts)

# --- The same as % of the year's households (unweighted) ---
share = (100 * counts.drop('Total households') / counts.loc['Total households']).round(1)
display(share)

# --- Binary target: % of households with ANY food insecurity (codes 1, 2, 3) ---
summary = pd.DataFrame({
    'households': hh.groupby('YEAR').size(),
    'unweighted_pct': (100 * hh.groupby('YEAR')['food_insecure'].mean()).round(1),    # % of households with any food insecurity
    # Weights each household by the major income earner's person weight FWEIGHT (the household weight we decided to use).
    # No bootstrap weights exist in the public file, so this is a point estimate without a design-based interval.
    'weighted_pct': pd.Series({
        y: 100 * np.average(g['food_insecure'], weights=g['FWEIGHT']) for y, g in hh.groupby('YEAR')}).round(1),
})
display(summary)
# index_label puts a name in the top-left cell of each CSV (without it that cell is empty)
counts.to_csv(f'{outputs_dir}/target_counts_by_year.csv', index_label='FSCHHLDM')
share.to_csv(f'{outputs_dir}/target_share_by_year.csv', index_label='FSCHHLDM')
summary.to_csv(f'{outputs_dir}/target_binary_by_year.csv', index_label='YEAR')

# --- Chart: shares of the four categories by year (stacked bars) ---
fig, ax = plt.subplots(figsize=(8, 4.5))
share.T.plot(kind='bar', stacked=True, ax=ax, colormap='viridis_r', edgecolor='white')
ax.set_ylabel('% of households (unweighted)')
ax.set_xlabel('CIS reference year')
ax.set_title('Household food security status by year')
ax.legend(title='FSCHHLDM', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
save_fig(fig, 'target_by_year')

## 2. Missing values and special codes

### Step 4 - How much is missing, by variable and year

**Purpose:** 

**What the code does:** 

**Learn from:** [DataFrame.pivot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html), [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html), [Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)

**My result:** 

In [ ]:
# Valid skip (question did not apply to this person) and not stated (applied but no answer) from notebook 01,
# counted for the major income earner of each kept household, before they were turned into missing values.
# The table covers the variables read from the files (ALFST is there only as the reason it was dropped).
miss = pd.read_csv(missing_file)
miss['missing_pct'] = miss['valid_skip_pct'] + miss['not_stated_pct'] + miss['blank_pct']
miss_wide = miss.pivot(index='variable', columns='YEAR', values='missing_pct').round(2)
display(miss_wide)
miss_wide.to_csv(f'{outputs_dir}/missing_pct_by_variable_year.csv')

# Cross-check on the saved table: missing values left in each predictor, as % of households, by year
display((100 * hh[predictors].isna().groupby(hh['YEAR']).mean()).round(2).T)

# Split the missing values for three variables into valid skip (question did not apply),
# not stated (applied but no answer) and blank. Uses the `miss` table built above.
# DataFrame.loc: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html
# Series.isin:   https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
cols_to_show = ['YEAR', 'variable', 'valid_skip_pct', 'not_stated_pct', 'blank_pct']
display(miss.loc[miss['variable'].isin(['ALFST', 'IMMST', 'HLEV2G']), cols_to_show])

## 3. Univariate: one predictor at a time

### Step 5 - Distribution of each predictor, by year

**Purpose:** 

**What the code does:** 

**Learn from:** [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html), [matplotlib Axes.barh](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html)

**My result:** 

In [ ]:
def univariate(var):
    """Share (%) of households in each category of `var`, for each year. Missing values are left out of the percentages."""
    tab = (100 * pd.crosstab(hh[var], hh['YEAR'], normalize='columns')).round(1)   # each year's column adds to 100
    tab.index = [label(var, c) for c in tab.index]
    tab.to_csv(f'{outputs_dir}/univariate_{var}.csv', index_label=var)
    display(tab)

    # grouped horizontal bars: one bar per year within each category
    fig, ax = plt.subplots(figsize=(8, 0.9 + 0.55 * len(tab)))
    height = 0.8 / len(years)
    for i, y in enumerate(years):
        ax.barh(np.arange(len(tab)) + i * height, tab[y], height=height, color=year_colors[i], label=str(y))
    ax.set_yticks(np.arange(len(tab)) + 0.4 - height / 2)
    ax.set_yticklabels([wrap(t) for t in tab.index], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('% of households (missing values excluded)')
    ax.set_title(f'{var}: distribution by year')
    ax.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left')
    save_fig(fig, f'univariate_{var}')

for var in predictors:
    print(f'\n--- {var} ---')
    univariate(var)

## 4. Bivariate: each predictor against the target

### Step 6 - Food insecurity rate by predictor category and year

**Purpose:** 

**What the code does:** 

**Learn from:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

**My result:** 

In [ ]:
def bivariate(var):
    """% of households with any food insecurity within each category of `var`, for each year.
    Two versions: unweighted (the households in the file) and weighted by FWEIGHT (comparable with published prevalence)."""
    g = hh.groupby([var, 'YEAR'])                     # one group per category and year; missing values are left out
    # .astype(float): a category that does not exist in some years becomes NaN, which the chart can skip;
    # pandas' own missing marker (<NA>) would make matplotlib fail.
    rate = (100 * g['food_insecure'].mean()).unstack('YEAR').astype(float).round(1)               # unweighted
    # Weighted rate = total weight of the food-insecure households / total weight of all households in the group
    rate_w = (100 * g['weight_x_insecure'].sum() / g['FWEIGHT'].sum()).unstack('YEAR').astype(float).round(1)
    n = g.size().unstack('YEAR')                                                                  # households behind each rate
    out = pd.concat({'unweighted_rate_pct': rate, 'weighted_rate_pct': rate_w, 'n_households': n}, axis=1)
    out.columns = [f'{stat}_{y}' for stat, y in out.columns]                                      # one flat header row
    out.index = [label(var, c) for c in out.index]
    out.to_csv(f'{outputs_dir}/bivariate_{var}.csv', index_label=var)
    display(out)

    # chart: the weighted rate (the population view, comparable with prevalence studies)
    fig, ax = plt.subplots(figsize=(8, 0.9 + 0.55 * len(rate_w)))
    height = 0.8 / len(years)
    for i, y in enumerate(years):
        ax.barh(np.arange(len(rate_w)) + i * height, rate_w[y], height=height, color=year_colors[i], label=str(y))
    ax.set_yticks(np.arange(len(rate_w)) + 0.4 - height / 2)
    ax.set_yticklabels([wrap(label(var, c)) for c in rate_w.index], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('% of households with any food insecurity (weighted by FWEIGHT)')
    ax.set_title(f'{var}: food insecurity rate by category and year')
    ax.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left')
    save_fig(fig, f'bivariate_{var}')

for var in predictors:
    print(f'\n--- {var} vs food insecurity ---')
    bivariate(var)

### Step 7 - How strong is each relationship, and does it change by year?

**Purpose:** 

**What the code does:** 

**Learn from:** [scipy chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html), [scipy association (Cramer's V)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html)

**My result:** 

In [ ]:
# For every predictor and year: chi-square test of independence and Cramer's V (0 = no association, 1 = perfect).
# Caution: these tests are unweighted and ignore the survey design. With tens of thousands of households almost every
# p-value is tiny, so read Cramer's V (the size of the association), not the p-value.
rows = []
for var in predictors:
    for y in years:
        sub = hh[hh['YEAR'] == y]
        tab = pd.crosstab(sub[var], sub['food_insecure'])         # counts; missing values are left out automatically
        try:
            chi2, p, dof, _ = chi2_contingency(tab)
            v = association(tab.values, method='cramer')
        except ValueError:                                         # e.g. only one category present in that year
            chi2, p, dof, v = np.nan, np.nan, np.nan, np.nan
        rows.append({'variable': var, 'YEAR': y, 'households_used': int(tab.values.sum()),
                     'chi2': round(chi2, 1), 'dof': dof, 'p_value': p, 'cramers_v': round(v, 3)})
tests = pd.DataFrame(rows)
tests.to_csv(f'{outputs_dir}/bivariate_tests.csv', index=False)

v_table = tests.pivot(index='variable', columns='YEAR', values='cramers_v')
display(v_table)
v_table.to_csv(f'{outputs_dir}/cramers_v_by_variable_year.csv')

# Chart: Cramer's V by year, one line per predictor (flat lines = a relationship of similar strength in every year)
fig, ax = plt.subplots(figsize=(8, 4.5))
for var in predictors:
    ax.plot(years, v_table.loc[var], marker='o', label=var)
ax.set_xticks(years)
ax.set_xlabel('CIS reference year')
ax.set_ylabel("Cramer's V with food insecurity")
ax.set_title('Strength of association with food insecurity, by year')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
save_fig(fig, 'cramers_v_by_year')

## 5. Relationships between predictors (training years 2018-2019)

### Step 8 - Which predictors overlap with each other?

**Purpose:** 

**What the code does:** 

**Learn from:** [scipy association (Cramer's V)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html), [matplotlib imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)

**My result:** 

In [ ]:
# Cramer's V between every pair of predictors, using the training years only (2018 and 2019).
train = hh[hh['YEAR'].isin([2018, 2019])]
pair = pd.DataFrame(np.nan, index=predictors, columns=predictors)
for a in predictors:
    for b in predictors:
        if a == b:
            pair.loc[a, b] = 1.0
        else:
            try:
                pair.loc[a, b] = association(pd.crosstab(train[a], train[b]).values, method='cramer')
            except ValueError:
                pass
pair = pair.round(2)
display(pair)
pair.to_csv(f'{outputs_dir}/predictor_association_2018_2019.csv', index_label='predictor')

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(pair.values.astype(float), vmin=0, vmax=1, cmap='viridis_r')
ax.set_xticks(range(len(predictors)))
ax.set_xticklabels(predictors, rotation=45, ha='right')
ax.set_yticks(range(len(predictors)))
ax.set_yticklabels(predictors)
for i in range(len(predictors)):
    for j in range(len(predictors)):
        ax.text(j, i, f'{pair.values[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if pair.values[i, j] > 0.5 else 'black')
fig.colorbar(im, ax=ax, label="Cramer's V")
ax.set_title("Association between predictors, CIS 2018-2019")
save_fig(fig, 'predictor_association_2018_2019')

print('Files saved in', outputs_dir)
print(sorted(os.listdir(outputs_dir)))